# Figure 1: closed-loop flutter suppression

A: experimental setup (`Setup.svg`, placed in Inkscape). B: real-time magnitude trace. C: FFT spectrum, uncontrolled vs. controlled. D: total power, paired.

In [ ]:
import glob, os
import cairosvg
import figurefirst as fifi
from IPython.display import display, SVG

from flutter.plotting import set_style
from flutter.config import OL_RDM_DIR, OUTPUT_DIR
from flutter.fft_energy import (process_condition_fft, draw_single_condition_fft_overlay,
                                draw_paired_reduction_panel, add_panel_label,
                                CONDITIONS, CLOSED_LOOP_PATTERN)

set_style()

In [ ]:
layout = fifi.svg_to_axes.FigureLayout('fig1.svg', autogenlayers=True, make_mplfigures=True, hide_layers=[])
print(layout.axes.keys())
print(layout.figures.keys())

In [ ]:
# Load all conditions from the OL_RDM bags
bag_dir = str(OL_RDM_DIR)

all_results = {}
for condition in CONDITIONS:
    if condition == 'CLOSED_LOOP':
        pattern = os.path.join(bag_dir, CLOSED_LOOP_PATTERN)
    else:
        pattern = os.path.join(bag_dir, f'{condition}_sample*.bag')
    bag_paths = sorted(glob.glob(pattern))
    if bag_paths:
        all_results[condition] = process_condition_fft(bag_paths, is_closed_loop=(condition == 'CLOSED_LOOP'))

In [ ]:
# Panel D: total power, paired
ax = layout.axes[('fig1', 'Total_power')]
draw_paired_reduction_panel(ax, all_results, condition='CLOSED_LOOP', scale=0.5,
                            controlled_color='#4CAF50', uncontrolled_color='grey',
                            show_legend=False)
fifi.mpl_functions.set_fontsize(ax, 6)

In [ ]:
# Panel C: FFT overlay
ax = layout.axes[('fig1', 'FFT')]
draw_single_condition_fft_overlay(ax, all_results, 'CLOSED_LOOP', scale=0.5,
                                  controlled_color='#4CAF50', uncontrolled_color='grey',
                                  show_legend=False, show_inset=True, title_color='black',
                                  inset_bbox=(0.42, 0.42, 0.54, 0.54),
                                  inset_freq_range=(3.5, 6.5))
fifi.mpl_functions.set_fontsize(ax, 6)

In [ ]:
# Panel B: real-time control trace
# TODO: needs flutter/time_trace.py (was plot_time_trace_updated.py, not added yet)
from flutter.time_trace import load_time_trace_data, draw_magnitude_panel

data = load_time_trace_data()

ax = layout.axes[('fig1', 'RT_control')]
draw_magnitude_panel(ax, data['json_time'], data['json_magnitudes'], data['json_labels'],
                     data['phase_offset_deg'], scale=0.8,
                     show_avg_labels=False, show_summary_box=False,
                     show_phase_label_box=False)
fifi.mpl_functions.set_fontsize(ax, 6)
ax.set_xlim(data['json_time'][0], data['json_time'][-1])
ax.set_ylim(0, max(data['json_magnitudes']) * 1.15)

In [ ]:
add_panel_label(layout.axes[('fig1', 'RT_control')], 'B', y=1.25, x=-0.03)
add_panel_label(layout.axes[('fig1', 'FFT')], 'C', x=-0.053)
add_panel_label(layout.axes[('fig1', 'Total_power')], 'D', x=-0.053)

In [ ]:
layout.append_figure_to_layer(layout.figures['fig1'], 'fig1', cleartarget=True)
out_svg = OUTPUT_DIR / 'fig1_filled.svg'
layout.write_svg(str(out_svg))

# figurefirst hides Inkscape layers 2 and 3 on write; turn them back on
svg_content = out_svg.read_text()
for layer in ['Layer 2', 'Layer 3']:
    svg_content = svg_content.replace(
        f'inkscape:label="{layer}" style="display:none"',
        f'inkscape:label="{layer}" style="display:inline"')
out_svg.write_text(svg_content)

display(SVG(str(out_svg)))

In [ ]:
cairosvg.svg2pdf(url=str(out_svg), write_to=str(OUTPUT_DIR / 'fig1_filled.pdf'))